# Notebook para treinamento do modelo

Transfer learning usando YAMNet como extrator de features.

**Saídas geradas:**
- `classifier_head.h5`: modelo Keras da cabeça treinada
- `classifier_head.onnx`: pesos exportados em ONNX (entregável do projeto)
- `training_report.json`: métricas de treino/teste


## 1. Instalar dependências

In [ ]:
!pip install -q tensorflow-hub librosa tf2onnx onnxruntime scikit-learn


## 2. Imports e configuração

In [ ]:
import os
import json
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices('GPU'))


In [ ]:
CLASSES = ["crying_baby", "dog"]
ESC50_URL = "https://github.com/karoldvl/ESC-50/archive/refs/heads/master.zip"
DATA_DIR = "esc50_data"
OUTPUT_DIR = "output"
SAMPLE_RATE = 16000
YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"
RANDOM_SEED = 42
EPOCHS = 30
BATCH_SIZE = 8

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


## 3. Baixar o ESC-50

In [ ]:
def baixar_esc50(data_dir: str = DATA_DIR) -> str:
    esc50_root = os.path.join(data_dir, "ESC-50-master")
    if os.path.isdir(esc50_root):
        print(f"[OK] ESC-50 já presente em {esc50_root}")
        return esc50_root

    os.makedirs(data_dir, exist_ok=True)
    zip_path = os.path.join(data_dir, "esc50.zip")

    print("Baixando ESC-50 (~600MB, pode levar alguns minutos)...")
    urllib.request.urlretrieve(ESC50_URL, zip_path)

    print("Extraindo...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(data_dir)

    os.remove(zip_path)
    print(f"[OK] ESC-50 pronto em {esc50_root}")
    return esc50_root


esc50_root = baixar_esc50()


In [ ]:
meta_completo = pd.read_csv(os.path.join(esc50_root, "meta", "esc50.csv"))
print("Categorias disponíveis no ESC-50:")
print(sorted(meta_completo["category"].unique()))

for classe in CLASSES:
    n = (meta_completo["category"] == classe).sum()
    status = "OK" if n > 0 else "NÃO ENCONTRADA"
    print(f"  '{classe}': {n} clipes [{status}]")


In [ ]:
def carregar_metadata(esc50_root: str) -> pd.DataFrame:
    meta_path = os.path.join(esc50_root, "meta", "esc50.csv")
    df = pd.read_csv(meta_path)
    df_filtrado = df[df["category"].isin(CLASSES)].reset_index(drop=True)
    print(f"[OK] {len(df_filtrado)} clipes encontrados para {CLASSES}")
    print(df_filtrado["category"].value_counts())
    return df_filtrado


df = carregar_metadata(esc50_root)
df.head()


## 4. Teste

In [ ]:
import IPython.display as ipd

for classe in CLASSES:
    subset = df[df["category"] == classe]
    assert len(subset) > 0, (
        f"Nenhum clipe encontrado para '{classe}'. "
    )
    exemplo = subset.iloc[0]
    caminho = os.path.join(esc50_root, "audio", exemplo["filename"])
    print(f"Classe: {classe} — {exemplo['filename']}")
    ipd.display(ipd.Audio(caminho))


## 5. Carregar o YAMNet (backbone pré-treinado, congelado)

In [ ]:
print("Carregando YAMNet pré-treinado do TensorFlow Hub...")
yamnet_model = hub.load(YAMNET_HANDLE)
print("[OK] YAMNet carregado (backbone congelado, não será re-treinado)")


## 6. Extrair embeddings de cada clipe

In [ ]:
def extrair_embedding(yamnet_model, caminho_audio: str) -> np.ndarray:
    audio, _ = librosa.load(caminho_audio, sr=SAMPLE_RATE, mono=True)
    audio = audio.astype(np.float32)

    _, embeddings, _ = yamnet_model(audio)   # embeddings: [n_frames, 1024]
    embedding_medio = tf.reduce_mean(embeddings, axis=0).numpy()
    return embedding_medio


def construir_dataset_embeddings(df: pd.DataFrame, esc50_root: str, yamnet_model):
    audio_dir = os.path.join(esc50_root, "audio")
    X, y = [], []

    for i, row in df.iterrows():
        caminho = os.path.join(audio_dir, row["filename"])
        try:
            emb = extrair_embedding(yamnet_model, caminho)
            X.append(emb)
            y.append(CLASSES.index(row["category"]))
        except Exception as e:
            print(f"  [AVISO] falhou em {row['filename']}: {e}")

        if (i + 1) % 10 == 0:
            print(f"  processados {i + 1}/{len(df)} clipes")

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


In [ ]:
cache_path = os.path.join(OUTPUT_DIR, "embeddings_cache.npz")

if os.path.exists(cache_path):
    print(f"[OK] Usando embeddings já extraídos em {cache_path}")
    dados = np.load(cache_path)
    X, y = dados["X"], dados["y"]
else:
    print("Extraindo embeddings YAMNet de cada clipe (feito uma única vez)...")
    X, y = construir_dataset_embeddings(df, esc50_root, yamnet_model)
    np.savez(cache_path, X=X, y=y)

print(f"[OK] Dataset de embeddings: X={X.shape}, y={y.shape}")


## 7. Split treino/teste

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
print(f"Treino: {len(X_train)} amostras | Teste: {len(X_test)} amostras")


## 8. Cabeça de classificação (a parte que de fato treinamos)

In [ ]:
def construir_cabeca_classificacao(input_dim: int = 1024, n_classes: int = 2) -> tf.keras.Model:
    modelo = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,), name="yamnet_embedding"),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(n_classes, activation="softmax", name="anomaly_probs"),
    ], name="crying_vs_bark_head")

    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo


modelo = construir_cabeca_classificacao(input_dim=X.shape[1], n_classes=len(CLASSES))
modelo.summary()


## 9. Treinar

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=6, restore_best_weights=True
)

historico = modelo.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=2,
)


In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(historico.history["loss"], label="treino")
plt.plot(historico.history["val_loss"], label="validação")
plt.title("Loss")
plt.xlabel("época")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(historico.history["accuracy"], label="treino")
plt.plot(historico.history["val_accuracy"], label="validação")
plt.title("Acurácia")
plt.xlabel("época")
plt.legend()

plt.tight_layout()
plt.show()


## 10. Avaliação

In [ ]:
y_pred = np.argmax(modelo.predict(X_test), axis=1)

relatorio = classification_report(y_test, y_pred, target_names=CLASSES, output_dict=True)
matriz_confusao = confusion_matrix(y_test, y_pred)

print(classification_report(y_test, y_pred, target_names=CLASSES))
print("Matriz de confusão:")
print(matriz_confusao)


In [ ]:
import seaborn as sns

plt.figure(figsize=(4, 4))
sns.heatmap(matriz_confusao, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de confusão")
plt.show()


## 11. Salvar modelo Keras e exportar para ONNX

In [ ]:
keras_path = os.path.join(OUTPUT_DIR, "classifier_head.h5")
modelo.save(keras_path)
print(f"[OK] Modelo Keras salvo em {keras_path}")


In [ ]:
import tf2onnx

def exportar_para_onnx(modelo, caminho_saida, input_dim=1024):
    input_signature = [tf.TensorSpec([None, input_dim], tf.float32, name="yamnet_embedding")]

    @tf.function(input_signature=input_signature)
    def modelo_fn(yamnet_embedding):
        return {"anomaly_probs": modelo(yamnet_embedding, training=False)}

    onnx_model, _ = tf2onnx.convert.from_function(
        modelo_fn, input_signature=input_signature, opset=13
    )

    with open(caminho_saida, "wb") as f:
        f.write(onnx_model.SerializeToString())

    print(f"[OK] Modelo exportado em ONNX: {caminho_saida}")


onnx_path = os.path.join(OUTPUT_DIR, "classifier_head.onnx")
exportar_para_onnx(modelo, onnx_path, input_dim=X.shape[1])


### 11.1. Validação o ONNX exportado


In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(onnx_path)
print("Input :", [(i.name, i.shape) for i in sess.get_inputs()])
print("Output:", [(o.name, o.shape) for o in sess.get_outputs()])

amostra = X_test[:3]
saida_onnx = sess.run(None, {"yamnet_embedding": amostra})[0]
saida_keras = modelo.predict(amostra)

print("\nSaída ONNX:\n", saida_onnx)
print("\nSaída Keras (deve bater com a de cima):\n", saida_keras)
print("\nDiferença máxima:", np.abs(saida_onnx - saida_keras).max())


## 12. Relatório final (para o relatório técnico da ponderada)

In [ ]:
relatorio_final = {
    "classes": CLASSES,
    "n_amostras_treino": len(X_train),
    "n_amostras_teste": len(X_test),
    "acuracia_final_teste": float(relatorio["accuracy"]),
    "classification_report": relatorio,
    "confusion_matrix": matriz_confusao.tolist(),
    "backbone": "YAMNet (google/yamnet/1, congelado)",
    "input_do_onnx": "embedding YAMNet de 1024 posições (média temporal)",
    "epochs_treinadas": len(historico.history["loss"]),
}

with open(os.path.join(OUTPUT_DIR, "training_report.json"), "w") as f:
    json.dump(relatorio_final, f, indent=2)

print(json.dumps(relatorio_final, indent=2))


## 13. Baixar os arquivos gerados

Roda a célula abaixo para baixar `classifier_head.onnx` (o entregável),
`classifier_head.h5` e `training_report.json` direto para sua máquina.


In [ ]:
from google.colab import files

files.download(os.path.join(OUTPUT_DIR, "classifier_head.onnx"))
files.download(os.path.join(OUTPUT_DIR, "classifier_head.h5"))
files.download(os.path.join(OUTPUT_DIR, "training_report.json"))
